# nb04 — Resultados principais: a curva custo × recall (H0)

**O que este notebook faz:** lê o gold humano da T22, o judge congelado nas duas configurações e
os scores N1/N2 da bateria de calibração, e produz:

1. a **curva custo × recall por camada** — `figures/fig05_custo_recall_h0.png`, a figura
   principal do trabalho;
2. o **recall por classe de falha (P, C, D)** — `figures/fig06_recall_por_classe_h0.png`, que é
   onde a predição de `ARQUITETURA §12` se verifica ou cai;
3. **`ΔRecall(N3 | N1+N2)` com IC bootstrap** — INS.2, o número que testa H0.

> **A aritmética não mora aqui.** Recall, falso alarme e o ganho incremental saem de
> `tapieval.scoring.ins`, que tem 16 testes. Um notebook que reimplementasse a conta produziria
> uma segunda versão dela, e a que aparece na figura seria justamente a que ninguém testou — é a
> mesma regra que o nb03 segue com o flip rate.

**O que este notebook NÃO mostra**, e cada limitação está repetida ao lado da figura que ela
afeta: n = 20 execuções; a metade determinística do gold é a saída do próprio detector (A27); e o
gold foi rotulado às cegas, então C2, C3 e C7 não existem nele.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from tapieval.labeling.cli import RotuloHumano
from tapieval.schema.trace import N3Judge
from tapieval.scoring.bateria import pontuar_bateria
from tapieval.scoring.ins import (
    CAMADAS,
    curva,
    falso_alarme,
    ganho_incremental,
    montar_item,
    recall_por_classe,
)

BATERIA = RAIZ / "runs" / "calibracao_2026-08-24"
JULGAMENTOS = RAIZ / "runs" / "judge_gold_calibracao_2026-08-24" / "julgamentos.jsonl"
ROTULOS = RAIZ / "labels" / "humano_2026-08-30.jsonl"
FIGURAS = RAIZ / "figures"

In [ ]:
%load_ext watermark
%watermark -u -d -v -m -p pandas,plotly,kaleido

---
## 1. O conjunto — e as três conferências que precedem qualquer número

O par que entra no recall é `(gold humano, detecção da camada)` **da mesma execução**. Três coisas
quebram isso sem quebrar nada:

- um rótulo da amostra de **melhoria** entrando no denominador (`METRICAS §5` proíbe: a fila é
  escolhida por dificuldade, e recall sobre casos difíceis não estima recall na população);
- uma execução julgada numa configuração e não na outra, que faria os dois pontos da curva terem
  denominadores diferentes;
- um rótulo apontando para um trace que não está nesta bateria — rotular contra uma bateria e
  julgar contra outra produz pares que nunca foram pares.

As três viram `assert` aqui, e não confiança em quem escreveu o notebook.

In [ ]:
scores = {s.run_id: s for s in pontuar_bateria(BATERIA).scores}

rotulos = {}
for linha in ROTULOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    rotulo = RotuloHumano.model_validate_json(linha)
    if rotulo.amostra == "estimativa":
        rotulos[rotulo.run_id] = rotulo

julgamentos: dict[str, dict[str, N3Judge]] = {}
for linha in JULGAMENTOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    registro = json.loads(linha)
    if registro.get("erro"):
        continue
    julgamentos.setdefault(registro["run_id"], {})[registro["configuracao"]] = (
        N3Judge.model_validate(registro["julgamento"])
    )

sem_score = [r for r in rotulos if r not in scores]
assert not sem_score, f"rótulo aponta para trace fora desta bateria: {sem_score}"

incompletos = [r for r in rotulos if set(julgamentos.get(r, {})) != {"cego", "com_trace"}]
assert not incompletos, (
    f"{len(incompletos)} execução(ões) sem as duas configurações — os dois pontos da curva "
    f"teriam denominadores diferentes: {incompletos[:3]}"
)

itens = [
    montar_item(rid, scores[rid].n1, scores[rid].n2, rot.para_n4humano(), julgamentos[rid])
    for rid, rot in sorted(rotulos.items())
]
print(f"{len(itens)} execuções de estimativa · {sum(len(i.gold) for i in itens)} falhas no gold")

---
## 2. O eixo x: custo por execução avaliada (INS.4)

O custo é medido em **tokens**, e não convertido em moeda. Preço de API muda, o número na figura
não deve mudar com ele — e a razão entre as camadas, que é o que a curva mostra, é a mesma nas
duas unidades. A camada determinística custa **zero token**: ela é função pura de
`(trace, gabarito)`, e `tests/test_repro.py` bloqueia `socket` para provar que continua sendo.

A mediana, e não a média: a distribuição de `tokens_in` do judge é assimétrica pelo tamanho da
evidência, e a média de um trace longo desloca a coluna inteira.

In [ ]:
custos = {"n1n2": 0.0}
por_config: dict[str, list[float]] = {"cego": [], "com_trace": []}
for linha in JULGAMENTOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    registro = json.loads(linha)
    if registro.get("erro"):
        continue
    custo = registro["custo"]
    por_config[registro["configuracao"]].append(custo["tokens_in"] + custo["tokens_out"])

custos["n1n2n3_cego"] = float(pd.Series(por_config["cego"]).median())
custos["n1n2n3_com_trace"] = float(pd.Series(por_config["com_trace"]).median())

tabela = pd.DataFrame(
    [
        {
            "camada": r.camada,
            "recall": r.valor,
            "acertos": f"{r.n_acertos}/{r.n_gold}",
            "falso_alarme": falso_alarme(itens, r.camada),
            "tokens_por_execucao": custos[r.camada],
            "identidade_A27": r.identidade,
        }
        for r in curva(itens)
    ]
).set_index("camada")
tabela

---
## 3. A figura principal — custo × recall

**Como ler.** Cada ponto é uma configuração do avaliador. Subir é detectar mais do gold; ir para a
direita é pagar mais por execução avaliada. A hipótese H0 prevê **retorno decrescente**: o
primeiro salto compra muito, o segundo compra pouco.

⚠️ **O primeiro ponto está marcado com contorno tracejado, e o motivo não é estético.** O recall
da camada determinística é **identidade, não medição** (A27): a metade P/D/C5 do gold é derivada
do mesmo `n1`/`n2` que a detecção. Lê-lo como "a camada barata já pega três quartos" é o erro que
o `METRICAS §11` existe para impedir. O que a figura mede de verdade é a **diferença** entre os
pontos — e é por isso que `METRICAS §7` marca a INS.2, e não a INS.1, como o número da hipótese.

In [ ]:
TINTA, TINTA2, SUPERFICIE = "#0b0b0b", "#52514e", "#fcfcfb"
AZUL, AMBAR, VERDE, CINZA = "#2a78d6", "#eda100", "#1baf7a", "#c9c8c4"

# O nome da camada é TICK do eixo x, e não anotação solta: anotação em `yref="paper"` cai
# em cima do título do eixo, e o overlap só aparece no PNG exportado — nunca no notebook.
ROTULO_DA_CAMADA = {
    "n1n2": "<b>N1+N2</b><br><sub>determinístico · 0 token</sub>",
    "n1n2n3_cego": "<b>+N3 cego</b><br><sub>judge vê a resposta</sub>",
    "n1n2n3_com_trace": "<b>+N3 com trace</b><br><sub>judge vê resposta e evidência</sub>",
}

x = [custos[c] for c in CAMADAS]
y = [tabela.loc[c, "recall"] for c in CAMADAS]

fig = go.Figure()
fig.add_scatter(
    x=x, y=y, mode="lines", line=dict(color=CINZA, width=3), showlegend=False, hoverinfo="skip"
)
# O ponto de identidade primeiro, com contorno tracejado; os dois medidos, sólidos.
fig.add_scatter(
    x=x[:1], y=y[:1], mode="markers+text", showlegend=False,
    marker=dict(size=20, color=SUPERFICIE, line=dict(color=TINTA2, width=3)),
    text=[f"  {y[0]:.1%}"], textposition="middle right",
    textfont=dict(color=TINTA2, size=13),
    hovertemplate="N1+N2 · recall %{y:.1%} · 0 token<extra></extra>",
)
fig.add_scatter(
    x=x[1:], y=y[1:], mode="markers+text", showlegend=False,
    marker=dict(size=20, color=AZUL, line=dict(color=AZUL, width=3)),
    text=[f"  {v:.1%}" for v in y[1:]], textposition="middle right",
    textfont=dict(color=AZUL, size=13),
    hovertemplate="recall %{y:.1%} · %{x:,.0f} tokens<extra></extra>",
)

fig.add_annotation(
    x=x[0], y=y[0], ax=30, ay=62, showarrow=True, arrowhead=0, arrowcolor=AMBAR, arrowwidth=2,
    text="<b>identidade, não medição</b><br><sub>A27 — a metade P/D do gold sai<br>do mesmo n1/n2 da detecção</sub>",
    font=dict(color=AMBAR, size=11), align="left", xanchor="left", yanchor="top",
)

ganho = ganho_incremental(itens)
fig.add_annotation(
    x=(x[0] + x[1]) / 2, y=(y[0] + y[1]) / 2, ax=60, ay=30, showarrow=True, arrowhead=0,
    arrowcolor=VERDE, arrowwidth=2, xanchor="left",
    text=(f"<b>ΔRecall = {ganho.delta:+.1%}</b><br>"
          f"<sub>IC95 [{ganho.ic95[0]:+.1%}, {ganho.ic95[1]:+.1%}] · INS.2</sub>"),
    font=dict(color=VERDE, size=12), align="left",
)

fig.update_layout(
    title=dict(
        text=("<b>O que cada camada de avaliação compra</b><br>"
              "<sub>recall contra o gold humano × tokens por execução avaliada · n = 20 · H0</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    xaxis=dict(
        title=dict(text="tokens por execução avaliada", standoff=26),
        gridcolor=CINZA, zeroline=False, range=[-900, max(x) * 1.30],
        tickmode="array", tickvals=x,
        ticktext=[ROTULO_DA_CAMADA[c] for c in CAMADAS],
    ),
    yaxis=dict(title="recall do gold", tickformat=".0%", gridcolor=CINZA,
               range=[0, 1.06], zeroline=False),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2),
    width=940, height=580, margin=dict(l=80, r=50, t=95, b=120),
)
fig.write_image(FIGURAS / "fig05_custo_recall_h0.png", scale=3)
fig.show()

---
## 4. Onde o ganho está — o recall por classe de falha

Esta é a figura que decide a hipótese, e não a de cima. `ARQUITETURA §12` não prevê apenas que o
recall sobe: prevê **onde** ele sobe. *"P e D são detectáveis sem LLM, com custo perto de zero; C
exige N3."*

E o documento é explícito sobre a refutação: se o ganho aparecesse **em todas as classes por
igual**, a estratificação estaria errada e o achado seria sobre a taxonomia, não sobre as camadas.
A curva agregada da seção 3 não distingue os dois casos. Esta distingue.

In [ ]:
por_classe = pd.DataFrame(
    {
        camada: {
            classe: r.valor for classe, r in recall_por_classe(itens, camada).items()
        }
        for camada in CAMADAS
    }
)
NOME_DA_CLASSE = {"P": "P — processo", "C": "C — conteúdo", "D": "D — decisão"}
tamanho_do_gold = {
    classe: recall_por_classe(itens, "n1n2")[classe].n_gold for classe in "PCD"
}

fig = go.Figure()
for camada, cor, nome in (
    ("n1n2", SUPERFICIE, "N1+N2 (determinístico)"),
    ("n1n2n3_cego", AZUL, "+N3 cego"),
):
    fig.add_bar(
        y=[NOME_DA_CLASSE[c] for c in "DPC"], x=[por_classe.loc[c, camada] for c in "DPC"],
        orientation="h", name=nome,
        marker=dict(color=cor, line=dict(color=TINTA2 if cor == SUPERFICIE else cor, width=2)),
        hovertemplate=nome + ": %{x:.0%}<extra></extra>",
    )

# O rótulo mostra ANTES → DEPOIS, e não só o delta. Em C o "antes" é 0%, e uma barra de
# comprimento zero é invisível: o leitor veria só a barra azul e concluiria que a camada
# determinística não foi medida ali — quando o que ela mediu foi exatamente zero, que é o
# achado. O número no texto é o que torna a barra ausente legível.
for i, classe in enumerate("DPC"):
    antes = por_classe.loc[classe, "n1n2"]
    depois = por_classe.loc[classe, "n1n2n3_cego"]
    fig.add_annotation(
        x=max(antes, depois) + 0.03, y=i, showarrow=False, xanchor="left",
        text=f"<b>{antes:.0%} → {depois:.0%}</b>"
        + f"<br><sub>{tamanho_do_gold[classe]} falhas no gold</sub>",
        font=dict(color=VERDE if depois > antes else CINZA, size=12), align="left",
    )

fig.update_layout(
    title=dict(
        text=("<b>O judge só compra conteúdo — processo e decisão o gabarito já dava</b><br>"
              "<sub>recall por classe de falha · n = 20 · a predição de H0, verificada<br>"
              "P e D em 100% são <b>identidade</b> (A27), não medição — o que se lê aqui é a "
              "linha do C</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    barmode="group", bargap=0.35,
    xaxis=dict(title="recall do gold", tickformat=".0%", gridcolor=CINZA, range=[0, 1.34]),
    yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.16, x=0),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2),
    width=940, height=500, margin=dict(l=150, r=50, t=110, b=90),
)
fig.write_image(FIGURAS / "fig06_recall_por_classe_h0.png", scale=3)
fig.show()

---
## 5. INS.2 — o número, e o que ele não cobre

`ΔRecall(N3 | N1+N2)` é uma **diferença**, e é por isso que ele sobrevive à limitação do A27: a
parte determinística é idêntica nos dois lados e cancela, sobrando a fração do gold que só o judge
alcança.

O IC é bootstrap percentil sobre reamostragem **das execuções** — a unidade de observação
independente é a run, não o código. Reamostrar códigos trataria dois códigos da mesma execução
como duas medidas independentes e devolveria um intervalo estreito demais, ou seja: o erro sairia
na direção que faz o achado parecer mais firme do que é.

In [ ]:
linhas = []
for de, para in (
    ("n1n2", "n1n2n3_cego"),
    ("n1n2n3_cego", "n1n2n3_com_trace"),
    ("n1n2", "n1n2n3_com_trace"),
):
    g = ganho_incremental(itens, de=de, para=para)
    linhas.append(
        {
            "de → para": f"{de} → {para}",
            "ΔRecall": g.delta,
            "IC95": f"[{g.ic95[0]:+.3f}, {g.ic95[1]:+.3f}]",
            "cruza zero?": "sim" if g.ic95[0] <= 0 <= g.ic95[1] else "não",
            "códigos ganhos": ", ".join(sorted(g.codigos_ganhos)) or "—",
        }
    )
pd.DataFrame(linhas).set_index("de → para")

### O segundo ponto de N3 não paga — e a frase honesta não é a óbvia

`cego → com_trace` dá **Δ = 0,0** com o IC cruzando zero, enquanto o custo quase dobra e o falso
alarme sobe de 1,0% para 3,7%. A leitura tentadora é *"dar o trace ao judge não acrescenta nada"*.

⚠️ **Ela é falsa, e o motivo está no gold.** O rotulador humano trabalhou **às cegas** — a CLI de
rotulagem impõe isso por construção, porque o κ da INS.6 exige que humano e judge vejam o mesmo
insumo. Logo `afirmacoes_sem_suporte`, `contradiz_evidencia` e `recomendou_acao_sem_base` vêm
`None` no gold, e **C2, C3 e C7 não existem nele**. São exatamente os três códigos que só o judge
com trace pode detectar: o que ele achar ali entra como **falso alarme**, nunca como acerto.

A frase que os dados sustentam é: **o gold disponível não tem como dizer se o trace acrescenta.**
Medir isso exigiria uma segunda rotulagem humana, com evidência à vista — e aí o κ precisaria dos
dois conjuntos. Fica declarado como limitação e como trabalho futuro, com o custo já dimensionado.

In [ ]:
resumo = {
    "n_execucoes": len(itens),
    "n_falhas_no_gold": sum(len(i.gold) for i in itens),
    "recall": {r.camada: round(r.valor, 4) for r in curva(itens)},
    "falso_alarme": {c: round(falso_alarme(itens, c), 4) for c in CAMADAS},
    "tokens_por_execucao": {c: round(custos[c], 1) for c in CAMADAS},
    "ins2_delta": round(ganho.delta, 4),
    "ins2_ic95": [round(v, 4) for v in ganho.ic95],
    "recall_por_classe": {
        camada: {classe: round(por_classe.loc[classe, camada], 4) for classe in "PCD"}
        for camada in CAMADAS
    },
}
(RAIZ / "docs" / "resultados_h0.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(resumo, indent=2, ensure_ascii=False))